# Aksara OCR — advanced experiments (items 8, 9, 10)

Three experiments that need training compute, drafted ready to run. Each section
is independent; run whichever you have quota for. Same setup and git-restore as
the other notebooks.

**Setup:** T4 x2, Internet On, **Save & Run All (Commit)**. Attach a previous
version's output to resume; each run skips what is already done.

**Cost overview (T4, 64px):**
- **Item 8 (learning curve):** 6 runs (~2.5 h). One session.
- **Item 9 (fine-tuned leakage):** 1 training run + one embedding pass (~40 min).
- **Item 10 (transfer):** the expensive one — 3 held-out scripts, each a
  ~12-script base training (~2-3 h) plus k-shot fine-tunes. Budget one session
  per held-out script (three sessions total).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Settings (right sidebar) > Accelerator > GPU T4 x2, then rerun."
)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
supported = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]

print(f"{name}  ({arch})")
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
print(f"torch {torch.__version__}  supports: {supported}")

# torch.cuda.is_available() returns True even when this build ships no kernels
# for the device - the failure then surfaces as a warning storm with every run
# landing in failures.jsonl. Check the architecture explicitly and stop here.
if arch not in supported:
    raise SystemExit(
        f"{name} is {arch}, but this PyTorch build only has kernels for "
        f"{supported}. Switch Settings > Accelerator to GPU T4 x2 (sm_75) "
        f"and rerun. The P100 is sm_60 and will not work."
    )

# Prove a real kernel runs, not just that a device is listed.
probe = (torch.randn(512, 512, device="cuda") @ torch.randn(512, 512, device="cuda")).sum()
torch.cuda.synchronize()
print(f"GPU compute OK (probe={probe.item():.1f})")

In [ ]:
# Internet must be ON (Settings > Internet) for these three lines.
import os
from pathlib import Path

REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"
REPO = Path("/kaggle/working/aksantara-ocr")

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Kaggle; installing the rest avoids a slow reinstall
# of torch against a possibly-mismatched CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"ready: {Path.cwd()}")

In [ ]:
# Fetch the cleaned, published v3. ~808 MB; needs Internet ON.
DOI = "10.17632/vfj32bpjsf.3"
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /kaggle/working/raw

RAW_ROOT = "/kaggle/working/raw" 

In [ ]:
# Pre-resize once to 224px. Source images run up to 1500x1500; decoding one
# costs ~6.7 ms/core, so without this the dataloader, not the GPU, is the limit.
# A 224px cache drops that to ~1 ms/img.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /kaggle/working/data --size 224

DATA_ROOT = "/kaggle/working/data"
# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}

In [ ]:
# Output dir for splits/manifest and all results (defined here so every
# experiment cell below can use it).
ARTIFACTS = "/kaggle/working/artifacts"

# stratified (no writer ids); --drop-duplicates removes images that became
# byte-identical after the 224px resize.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" --out-dir "{ARTIFACTS}" \
    --split-strategy stratified --drop-duplicates

# Verify against published v3. Image/class/script counts come from manifest.csv,
# written before any filtering, so they are invariant.
import pandas as pd
manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {"images": len(manifest), "classes": manifest["label"].nunique(),
          "scripts": manifest["script"].nunique()}
for k, want in EXPECTED.items():
    print(f"  {k:8} {actual[k]:6}  expected {want:6}  {'OK' if actual[k]==want else 'MISMATCH'}")
if actual != EXPECTED:
    raise SystemExit("Data does not match published v3 — re-run fetch and cache cells.")
print("Matches published v3.")

## Item 8 — Learning curve
Train ResNet-18 @64px on 5 / 10 / 25 / 50 / 75 / 100 % of the training set. Shows
how much data the task needs and argues for the corpus's size. Each fraction
writes to its own results dir; the last cell aggregates.

In [ ]:
# Learning curve — one run per train fraction (resnet18, unified, 64px, seed 0).
import subprocess, re
FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75, 1.00]
LC = f"{ARTIFACTS}/results/learning_curve"
for frac in FRACTIONS:
    rd = f"{LC}/frac_{int(frac*100):03d}"
    print(f"\n=== train fraction {frac:.0%} -> {rd} ===")
    !python scripts/02_run_matrix.py --config configs/single_resnet18.yaml \
        --artifacts "{ARTIFACTS}" --results "{rd}" \
        --num-workers 4 --time-budget 8 --train-fraction {frac} 2>&1 | grep -E "train-fraction|run:|acc=|complete|FAILED|Error"

In [ ]:
# Aggregate: macro-F1 vs training-set size.
import json, glob, pandas as pd
rows=[]
for f in sorted(glob.glob(f"{ARTIFACTS}/results/learning_curve/frac_*/resnet18__unified*/result.json")):
    d=json.load(open(f)); frac=int(f.split("frac_")[1][:3])
    rows.append({"train_pct":frac, "n_train":d["n_train"],
                 "macro_f1":d["test_metrics"]["macro_f1"]*100,
                 "accuracy":d["test_metrics"]["accuracy"]*100})
if rows:
    print(pd.DataFrame(rows).sort_values("train_pct").to_string(index=False))
else:
    print("No learning-curve result.json found — the training runs failed.")
    import glob as _g
    for fj in _g.glob(f"{ARTIFACTS}/results/learning_curve/frac_*/failures.jsonl"):
        print(open(fj).read()[:1500])

## Item 9 — Fine-tuned-embedder leakage re-run
Train one ResNet-18 with `--save-checkpoints`, then rerun the near-duplicate
audit (§7a of RESULTS.md) with that fine-tuned penultimate layer — the ideal,
geometry-invariant embedder. Upgrades the audit from HOG lower-bound to
definitive.

In [ ]:
# 1. Train one resnet18 @64px, saving the checkpoint.
CKPT_DIR = f"{ARTIFACTS}/results/ckpt"
!python scripts/02_run_matrix.py --config configs/single_resnet18.yaml \
    --artifacts "{ARTIFACTS}" --results "{CKPT_DIR}" \
    --num-workers 4 --time-budget 4 --save-checkpoints 2>&1 | grep -E "run:|acc=|complete|FAILED|Error"

# 2. Locate the checkpoint and run the audit with the fine-tuned embedder.
import glob
ckpt = sorted(glob.glob(f"{CKPT_DIR}/resnet18__unified*/model.pth"))[0]
print("checkpoint:", ckpt)
!python scripts/07_leakage_audit.py --artifacts "{ARTIFACTS}" \
    --embedder resnet18 --checkpoint "{ckpt}" --image-size 64

## Item 10 — Leave-one-script-out transfer (highest ceiling)
Does the corpus work as a pretraining resource for Indonesian scripts *not* in
it? For each held-out script: pretrain on the other 12, fine-tune with
10/50/100 examples/class, compare against an ImageNet start. The transfer−imagenet
gap at small k is the dataset's value as a pretraining resource.

Three held-out scripts span the range: **Lontara** (138-class syllabary),
**Lampung** (20-class alphabet), **Jawi** (hard). **Run one per session** — the
base training dominates. Each writes to `results/transfer_<script>`.

In [ ]:
# Held-out: Lontara (large syllabary). One session (base training ~2-3 h + fine-tunes).
!python scripts/09_transfer_experiment.py --artifacts "{ARTIFACTS}" \
    --held-out Lontara --shots 10 50 100 \
    --epochs-base 30 --epochs-ft 40 --image-size 64

In [ ]:
# Held-out: Lampung (compact alphabet). One session (base training ~2-3 h + fine-tunes).
!python scripts/09_transfer_experiment.py --artifacts "{ARTIFACTS}" \
    --held-out Lampung --shots 10 50 100 \
    --epochs-base 30 --epochs-ft 40 --image-size 64

In [ ]:
# Held-out: Jawi (hard). One session (base training ~2-3 h + fine-tunes).
!python scripts/09_transfer_experiment.py --artifacts "{ARTIFACTS}" \
    --held-out Jawi --shots 10 50 100 \
    --epochs-base 30 --epochs-ft 40 --image-size 64

In [ ]:
# Summary across held-out scripts (run after all three finish).
import pandas as pd, glob
frames=[pd.read_csv(f) for f in glob.glob(f"{ARTIFACTS}/results/transfer_*/transfer_results.csv")]
if frames:
    df=pd.concat(frames)
    piv=df.pivot_table(index=["held_out","k"], columns="init", values="macro_f1")
    piv["gain"]=piv["transfer"]-piv["imagenet"]
    print(piv.round(2).to_string())
else:
    print("no transfer results yet")

## After running
Output persists with each committed version. Back up metrics to git as before
(result.json / CSVs, not the .pth or npz). Paste the tables here and I will fold
items 8/9/10 into RESULTS.md with figures.